
# 3D Image Segmentation Pipeline – Step 05: Quantitative Analysis

## Description
This notebook extracts **quantitative measures** from the final label maps:
- **Volume** (voxel count) per labeled region
- **Center of Mass (CoM)** in voxel coordinates
- **Equivalent diameter** assuming a perfect sphere

## Outputs
- Tabulated metrics per label (volume, CoM, equivalent diameter)
- Optional CSV export for downstream analysis

## Part of Pipeline
- **Previous Step**: Watershed Segmentation
- **Next Step**: N/A


In [19]:
# Import required libraries
import numpy as np

# Integrate figures into notebook
%matplotlib notebook

### Load the labeled, i.e. segmented, image from the previous step


In [20]:
# Load binary image
labeled_image = np.load(file='data/labeled_image.npy')


### Step 1 — Collect unique labels

We collect unique label IDs from the 3D label maps and **exclude background (0)**.



In [21]:
# Custom
labels = np.unique(labeled_image)
print("Labels:", labels.tolist())


Labels: [0, 1, 2, 3]


### Step 2 — Index grids

We create index arrays `(z_idx, y_idx, x_idx)` matching the label map shape  so we can compute CoM by averaging index positions where each label occurs

In [22]:
# Index grids (z, y, x) for the custom map
z_idx_c, y_idx_c, x_idx_c = np.indices(labeled_image.shape)

# Index grids for the library map
z_idx_l, y_idx_l, x_idx_l = np.indices(labeled_image.shape)

print("Shape check (custom):", labeled_image.shape)
print("Shape check (library):", labeled_image.shape)


Shape check (custom): (16, 16, 16)
Shape check (library): (16, 16, 16)


### Step 3 — Volume and CoM for custom segmentation

For each label:
- `mask = (labels_3d == lab)`
- `voxel_count = mask.sum()`
- `com_z = z_idx[mask].sum() / voxel_count`, likewise for `com_y`, `com_x`.


In [23]:
# Prepare result arrays
volume = np.zeros(labels.shape, dtype=int)
CoM = np.zeros((labels.size, 3), dtype=float)  # (z, y, x)

# Loop over labels (custom)
for i, lab in enumerate(labels):
    mask = (labeled_image == lab)
    count = int(mask.sum())
    volume[i] = count

    if count > 0:
        com_z = z_idx_c[mask].sum() / count
        com_y = y_idx_c[mask].sum() / count
        com_x = x_idx_c[mask].sum() / count
        CoM[i] = (com_z, com_y, com_x)
    else:
        CoM[i] = (np.nan, np.nan, np.nan)

print("Custom — volumes:", volume.tolist())
print("Custom — CoM (z,y,x) sample:", CoM[:min(3, len(CoM))])


Custom — volumes: [3724, 1, 253, 118]
Custom — CoM (z,y,x) sample: [[ 7.51825994  7.62298604  7.72046187]
 [ 5.          6.         14.        ]
 [ 6.99209486  4.96047431  3.96047431]]


### Step 5 — Equivalent diameter

We derive the **equivalent diameter** assuming a sphere with the same volume: $d_\text{eq} = \left(\frac{6\,V}{\pi}\right)^{1/3}$ (Units: voxels)


In [24]:
equiv_diameter     = ((6.0 * volume     / np.pi) ** (1.0 / 3.0))

### Step 6 — Tables

We print **label**, **volume**, **CoM (z, y, x)** (rounded for display), and **eq. diameter**.


In [25]:
print("Label | Volume (voxels) | COM (z, y, x)           | Eq. Diameter (voxels)")
print("----------------------------------------------------------------------")
for i, lab in enumerate(labels):
    z, y, x = CoM[i]
    com_str = f"({z:.0f}, {y:.0f}, {x:.0f})"
    print(f"{lab:5d} | {volume[i]:15d} | {com_str:25s} | {equiv_diameter[i]:.2f}")


Label | Volume (voxels) | COM (z, y, x)           | Eq. Diameter (voxels)
----------------------------------------------------------------------
    0 |            3724 | (8, 8, 8)                 | 19.23
    1 |               1 | (5, 6, 14)                | 1.24
    2 |             253 | (7, 5, 4)                 | 7.85
    3 |             118 | (8, 9, 8)                 | 6.09



**Exercise 1:** Label Count Consistency
   - **Task:** Compare the number of labels (excluding background) to the number of spheres you generated in Step 01.
   - **Questions:** If counts differ, why? (e.g., overlapping spheres, merged watershed basins, missing seeds)

**Exercise 2:** Sphere Sanity: Diameter vs Known Radius
   - **Task:** For spherical segments, check whether `Eq. Diameter ≈ 2 × known radius` (voxel units).
   - **Questions:** Where do you see deviations? Consider discretization, partial overlap, boundary leakage.


**Exercise 3:** CoM Validation on Slices
   - **Task:** Verify Center of Mass `(z, y, x)` by plotting crosshairs or markers on the corresponding slice.
   - **Questions:** Do CoMs lie near the visual center of each object?